In [ ]:
# Import dependencies

import torch
from torch import nn
from torch.utils.data import DataLoader
import numpy as np
import ehtim as eh
import os

# Local imports
import data.imgTransforms as imgTransforms
import data.dataset_img as ds
import data.CI_torch_v2 as CI
from models.encoder import Encoder
from models.decoder import Decoder
import models.unet2dcondition as unet



In [ ]:
# Configuration Parameters

batch_size = 32 
model_name = 'test_GenDIReCT'
ae_path = 'models/saved_models/Autoencoders/DIReCT_v2_AEweights.pt'
uvfits_path = 'uvfits/SR1_M87_2017_101_hilo_hops_netcal_StokesI.uvfits'
data_dir = 'data/sample_training_data/'
baseid = 0
fov = 225  # Field of view in muas

# Device setup
device = "mps" if torch.backends.mps.is_available() else ("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Fixed to 1 epoch
epochs = 1
learning_rate = 1e-3

print(f"Model name: {model_name}")
print(f"Batch size: {batch_size}")
print(f"Learning rate: {learning_rate}")
print(f"Device: {device}")

In [ ]:
# Helper functions

def load_dataset(data_dir, mnames, transforms):
    """Load dataset from .npy files"""
    filenames = np.array([data_dir + 'imgs_' + m + '.npy' for m in mnames])
    dataset = ds.ImgDataset(filenames, transform_list=transforms)
    return dataset

def count_parameters(model):
    """Count trainable parameters in model"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def load_encoder(encoder_path, device):
    """Load pre-trained encoder and decoder"""
    encoder = Encoder(1, 4, n_res_layers=2, res_h_dim=8).to(device)
    decoder = Decoder(4, 4, n_res_layers=2, res_h_dim=8).to(device)
    state_dict = torch.load(encoder_path, map_location=device)
    encoder_state_dict = {k.replace('encoder.', ''): v for k, v in state_dict.items() if 'encoder' in k}
    decoder_state_dict = {k.replace('decoder.', ''): v for k, v in state_dict.items() if 'decoder' in k}
    encoder.load_state_dict(encoder_state_dict)
    decoder.load_state_dict(decoder_state_dict)
    return encoder, decoder


In [ ]:
# Load obs and CI evaluator

imgdim = 64

# Load observation data
obs = eh.obsdata.load_uvfits(uvfits_path)

# Calculate pixel scale
psize = fov * 1e-6 / 206265 / imgdim

# Create closure invariants object
clObj = CI.Closure_Invariants(obslist=[obs], device=device, ttype='DFT', 
                                baseid=baseid, psize=psize)

# Infer data dimension from closure invariants
im = np.zeros((1, imgdim, imgdim))
im[0, imgdim//2, imgdim//2] = 1.0
data_dim = clObj.FTCI(im).shape[-1]

print(f"Image dimension: {imgdim}x{imgdim}")
print(f"Pixel scale: {psize:.6e} rad")
print(f"Number of closure invariants: {data_dim}")
print("Closure invariants object loaded!")

In [ ]:
# Load pre-trained encoder and decoder
encoder, decoder = load_encoder(ae_path, device)
encoder.eval()  # Set to evaluation mode
decoder.eval()

# Freeze encoder parameters - we only need it for encoding
for param in encoder.parameters():
    param.requires_grad = False
    
print(f"Decoder loaded with {count_parameters(decoder):,} parameters")

In [ ]:
# Initialise GenDIReCT model
model = unet.UNet2DCondition(ci_dim=data_dim, model_choice=1, encoder_hid_dim=0).to(device).double()

# Try to load existing checkpoint if it exists
model_save_path = f'models/saved_models/{model_name}.pt'
if os.path.exists(model_save_path):
    print(f"Loading existing model from {model_save_path}")
    model.load_state_dict(torch.load(model_save_path, map_location=device))
else:
    print(f"No existing checkpoint found at {model_save_path}. Creating fresh model.")

# Create optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Loss function
loss_fn = nn.MSELoss()

print(f"Model has {count_parameters(model):,} trainable parameters")
print(f"Learning rate: {learning_rate}")

In [ ]:
# Dataset and DataLoader setup

# Set up transforms
tforms = imgTransforms.imgTransforms()

# Choose which datasets to use
mnames = ['gauss', 'disk', 'ellipse', 'ring']
transforms_list = [tforms.train_transforms] * (len(mnames))

# Load dataset
dataset = load_dataset(data_dir, mnames, transforms_list)

# Create dataloader with reduced num_workers for notebooks
train_dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, 
                              num_workers=0, pin_memory=True)

print(f"Dataset loaded with {len(dataset)} images")
print(f"Number of batches per epoch: {len(train_dataloader)}")
print(f"Training datasets: {mnames}")

# Plot random assortment of augmented images from the dataset
import matplotlib.pyplot as plt
def plot_random_images(dataset, num_images=9):
    """Plot a random assortment of images from the dataset"""
    indices = np.random.choice(len(dataset), num_images, replace=False)
    fig, axes = plt.subplots(1, num_images, figsize=(15, 5))
    for ax, idx in zip(axes, indices):
        img, _ = dataset[idx]
        ax.imshow(img.squeeze(), cmap='gray')
        ax.axis('off')
    plt.show()

plot_random_images(dataset, num_images=9)


In [ ]:
# Training step

def train_epoch(model, encoder, clObj, device, dataloader, optimizer, epoch, loss_fn=nn.MSELoss()):
    """Train for one epoch"""
    model.train()
    noise_scheduler = model.scheduler
    losses = []
    
    print(f"\nEpoch {epoch+1}/{epochs} Training:")
    print("-" * 50)
    
    for batch_idx, (x, y) in enumerate(dataloader):
        # Compute closure invariants
        ci = clObj.FTCI(x, train=True)
        
        # Encode images to latent space
        with torch.no_grad():
            x_encoded = encoder(x.to(device))
        
        # Prepare closure invariants
        ci = ci.unsqueeze(1).to(device)
        
        # Sample random noise and timesteps
        noise = torch.randn_like(x_encoded, dtype=torch.float64)
        timesteps = torch.randint(0, 999, (x_encoded.shape[0],)).long().to(device)
        
        # Add noise using scheduler
        noisy_x = noise_scheduler.add_noise(x_encoded, noise, timesteps)
        
        # Predict noise
        pred = model(noisy_x, timesteps, ci)
        loss = loss_fn(pred, noise)
        
        if not torch.isnan(loss):
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            losses.append(loss.item())
            
            if (batch_idx + 1) % 10 == 0 or batch_idx == 0:
                print(f"  Batch {batch_idx+1:3d}/{len(dataloader)}, Loss: {loss.item():.6f}")
    
    # Compute average loss
    avg_loss = np.mean(losses) if losses else 0
    print(f"\nEpoch {epoch+1} Summary:")
    print(f"  Average Loss: {avg_loss:.6f}")
    print(f"  Total Batches: {len(dataloader)}")
    print("-" * 50)
    
    return avg_loss



In [ ]:
# Execute training loop
import time

start_time = time.time()

# Run training for 1 epoch
avg_loss = train_epoch(model, encoder, clObj, device, train_dataloader, optimizer, epoch=0, loss_fn=loss_fn)

elapsed_time = time.time() - start_time

print(f"Final Average Loss: {avg_loss:.6f}")
print(f"Time Elapsed: {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")


In [ ]:
# Saving trained model weights
save_dir = 'models/saved_models'
os.makedirs(save_dir, exist_ok=True)

# Save model
model_path = os.path.join(save_dir, f'{model_name}.pt')
torch.save(model.state_dict(), model_path)

print(f"Model saved to: {model_path}")
print(f"Total model parameters: {count_parameters(model):,}")